# IronVeil — re-corrida de C0-C6 en Colab GPU (V1-V4)

**Objetivo:** que C0-C6 queden medidas con el MISMO código del proxy (calentamiento, keep_alive, conexión reutilizada) y en el mismo hardware (GPU T4 de Colab), para que la latencia de la tabla maestra deje de mezclar CPU con GPU. Decisión ya tomada: las filas viejas de C0-C5 (CPU) se descartan y se reemplazan por las de esta corrida.

**Antes de empezar:** Entorno de ejecución → Cambiar tipo de entorno → **T4 GPU**. Luego ejecuta las celdas en orden.

**Seguridad:** `HF_TOKEN`, `SPT_SECRET` y `RRHH_SECRET` los escribes TÚ en la celda 4 (`getpass`). Nunca los pegues en una celda como texto, ni en un chat.

**Alcance:** V1-V4 (lo que entra en la métrica de latencia; V5 ya está excluido de esa métrica por diseño).

## 1. Verificar GPU y clonar el repo (rama `develop`)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
%cd /content
!test -d IronVeil || git clone https://github.com/Anderfg13/IronVeil.git
%cd /content/IronVeil
!git checkout develop
!git pull

## 2. Instalar Ollama (`zstd` primero, lo exige el instalador)

In [ ]:
!apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

## 3. Arrancar el servidor de Ollama (con `keep_alive` de 30 min)

In [ ]:
import os, subprocess, time

env_ollama = {**os.environ, "OLLAMA_KEEP_ALIVE": "30m"}
ollama_proc = subprocess.Popen(
    ["ollama", "serve"], env=env_ollama,
    stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT,
)
for _ in range(30):
    if subprocess.run(["ollama", "list"], capture_output=True).returncode == 0:
        break
    time.sleep(2)
!ollama list

## 4. Secretos y variables de entorno

Te va a pedir 3 valores con `getpass` (no se muestran al escribirlos). `BASE_MODEL` debe ser **el mismo** que se usó en la corrida de referencia de C6, para que las 7 configuraciones sean comparables.

In [ ]:
import getpass, os

os.environ["HF_TOKEN"] = getpass.getpass("HF_TOKEN: ")
os.environ["SPT_SECRET"] = getpass.getpass("SPT_SECRET: ")
os.environ["RRHH_SECRET"] = getpass.getpass("RRHH_SECRET: ")

os.environ.update({
    "BASE_MODEL": "llama3.2:1b",
    "MODELO_CLASIFICADOR": "llama-guard3:1b",
    "MODELO_PROMPT_GUARD_ID": "meta-llama/Llama-Prompt-Guard-2-86M",
    "OLLAMA_HOST": "http://localhost:11434",       # lo usa ollama/init.sh
    "OLLAMA_BASE_URL": "http://localhost:11434",   # lo usa el proxy
    "OLLAMA_KEEP_ALIVE": "30m",
})
print("Variables listas (secretos no se imprimen).")

## 5. Crear los modelos `soporte` y `rrhh` + descargar el clasificador

In [ ]:
# init.sh espera los Modelfile en /modelfiles (ruta de Docker); en Colab es un symlink.
!ln -sfn /content/IronVeil/ollama/modelfiles /modelfiles
!bash ollama/init.sh
!ollama list

## 6. Dependencias del proxy

In [ ]:
!pip install -q -r proxy/requirements.txt

## 7. Arrancar el proxy (con el clasificador activo para que se **precaliente**)

El proxy precalienta `soporte`, `rrhh` y —solo si `clasificacion` está en `true` al arrancar— Prompt Guard y Llama Guard. Por eso se arranca con la bandera de C3: así la carga en frío ocurre ahora y NO contamina la latencia de la primera petición medida. Después, `config.yaml` se reescribe por configuración (el proxy lo lee en cada petición, no hace falta reiniciarlo).

Esta celda puede tardar varios minutos la primera vez (descarga/carga de Prompt Guard).

In [ ]:
import subprocess, time, urllib.request

REPO = "/content/IronVeil"
FLAGS = ["filtrado", "delimitacion", "clasificacion", "minimo_privilegio", "aprobacion_humana"]
CONFIGS = {
    "C0": [],
    "C1": ["filtrado"],
    "C2": ["delimitacion"],
    "C3": ["clasificacion"],
    "C4": ["minimo_privilegio"],
    "C5": ["aprobacion_humana"],
    "C6": list(FLAGS),
}

def escribir_config(nombre):
    activos = CONFIGS[nombre]
    contenido = "\n".join(f"{f}: {'true' if f in activos else 'false'}" for f in FLAGS) + "\n"
    with open(f"{REPO}/config.yaml", "w") as fh:
        fh.write(contenido)
    return contenido

escribir_config("C3")  # clasificacion=true => el calentamiento carga los clasificadores
proxy_proc = subprocess.Popen(
    ["uvicorn", "proxy.main:app", "--host", "0.0.0.0", "--port", "8000", "--no-server-header"],
    cwd=REPO, stdout=open("/content/proxy.log", "w"), stderr=subprocess.STDOUT,
)

inicio = time.time()
while time.time() - inicio < 900:
    try:
        if urllib.request.urlopen("http://localhost:8000/health", timeout=3).status == 200:
            break
    except Exception:
        time.sleep(5)
else:
    raise RuntimeError("El proxy no respondio en 15 min; revisa /content/proxy.log")

print(f"Proxy listo en {time.time() - inicio:.0f}s")
!grep -i "calentamiento" /content/proxy.log
!ollama ps

**Revisa la salida de `ollama ps`:** la columna `PROCESSOR` debe decir `100% GPU`. Si dice `CPU`, algo está mal (Ollama no está usando la GPU) y la corrida no cumple su objetivo.

> ⚠️ **Aviso sobre C5 y C6 (aprobación humana):** el límite de tasa del mecanismo (10 peticiones/min por cliente, 50/min global) es fijo. En GPU las peticiones salen mucho más rápido que en CPU, así que en C5 el propio tráfico de la batería puede saturar el límite y marcar como bloqueadas peticiones que no lo estaban por contenido — el mismo efecto ya documentado en la corrida de referencia de C6 (`docs/FUENTE_DE_VERDAD.md`, 2026-09-30). Los datos de C5 y C6 de esta corrida deben leerse con esa salvedad. (La celda 8 espera 65 s antes de cada C5/C6 para que cada repetición parta de la ventana vacía.)

## 8. Correr V1-V4 en C0 → C6 (3 repeticiones)

In [ ]:
import os, subprocess

open("/tmp/inicio_bateria", "w").close()   # marca de tiempo: solo se empaquetan los JSONL nuevos
CONFIGS_A_CORRER = ["C0", "C1", "C2", "C3", "C4", "C5", "C6"]   # editar para repetir solo alguna
REPETICIONES = 3   # n por celda es chico (5-21 intentos); mas repeticiones = menos ruido
SCRIPTS = ["ataques/vectores_1_2_3.py", "ataques/vector4_movimiento_lateral.py"]
env = {**os.environ, "PYTHONPATH": REPO}

for rep in range(1, REPETICIONES + 1):
    for nombre in CONFIGS_A_CORRER:
        print("=" * 70)
        print(f"Repeticion {rep}/{REPETICIONES} - {nombre}: config.yaml ->")
        print(escribir_config(nombre))
        if "aprobacion_humana" in CONFIGS[nombre]:
            # Deja que se vacie la ventana de 60s del rate limit, para que cada
            # repeticion de C5/C6 arranque desde el mismo estado.
            time.sleep(65)
        for script in SCRIPTS:
            r = subprocess.run(
                ["python", script, "--configuracion", nombre, "--url-proxy", "http://localhost:8000"],
                cwd=REPO, env=env, capture_output=True, text=True,
            )
            print(f"--- {script} (exit {r.returncode})")
            print((r.stdout + r.stderr)[-800:])
            if r.returncode != 0:
                print(f"!!! {script} FALLO en {nombre}; revisa la salida de arriba antes de seguir.")

## 9. Empaquetar los resultados nuevos (JSONL de los ataques + `eventos.jsonl` del proxy)

In [ ]:
import glob, os, zipfile

marca = os.path.getmtime("/tmp/inicio_bateria")
nuevos = sorted(
    p for p in glob.glob(f"{REPO}/resultados/**/*.jsonl", recursive=True)
    if os.path.getmtime(p) >= marca
)
destino = "/content/resultados_gpu_c0_c6.zip"
with zipfile.ZipFile(destino, "w", zipfile.ZIP_DEFLATED) as z:
    for p in nuevos:
        z.write(p, os.path.relpath(p, REPO))

for p in nuevos:
    print(os.path.relpath(p, REPO))
print(f"\n{len(nuevos)} archivos -> {destino} ({os.path.getsize(destino)/1024:.0f} KB)")

## 10. Descargar

1. Panel de archivos (ícono de carpeta, izquierda) → `/content/resultados_gpu_c0_c6.zip` → clic derecho → **Descargar**. (`files.download()` desde una celda ya falló antes en silencio; este camino sí funciona.)
2. Si aparece el diálogo nativo de "Guardar como" de Windows, guárdalo en la carpeta scratchpad de la sesión de Claude Code.
3. Avísale a Claude: él descarta las filas viejas de C6 (otra versión del código) a `descartados.csv` con razón documentada, ingesta las nuevas y regenera tabla maestra, gráficas y `resultados_finales.md`.

In [ ]:
# Opcional, al terminar: apagar procesos
proxy_proc.terminate()
ollama_proc.terminate()